# ITEC-Rep Customer Cleansing — `Customer2` (BNN · Studio7 retail POS)

**เป้าหมาย:** ทำความสะอาดลูกค้า ITEC-Rep และเช็กซ้ำ*ภายใน* ITEC ด้วยกฎชุดเดียวกับ CRM → ได้จำนวนลูกค้าจริงที่ไม่ซ้ำ
และรู้ว่าแต่ละคนมีคีย์อะไรพร้อมใช้ map ข้ามฐาน (เลขบัตร · มือถือ · อีเมล · CRMID)

| ขั้น | ทำอะไร | ตัดแถวออกไหม |
|---|---|---|
| Extract | ดึงทีละ 100,000 แถวเรียงตาม `CusCode` (clustered PK) · `WITH (NOLOCK)` · เลือกเฉพาะคอลัมน์ที่ใช้ | – |
| Name Standardization | ตัดคำนำหน้า · แยกชื่อ-นามสกุลจากช่องเดียว · นามสกุลที่เป็นแค่ `.` `-` = ว่าง · แยกไทย/อังกฤษ | ไม่ตัด (แก้ค่า) |
| Contact Validation | เบอร์ (รวมช่องที่มีหลายเบอร์ → ใช้เบอร์แรก) · อีเมล · วันเกิด · **เลขบัตร `CitizenNo` / `TaxNo` + checksum** | ไม่ตัด (ค่าที่ใช้ไม่ได้ = ว่าง) |
| Record Classification | ไม่มีข้อมูลระบุตัว · บัญชีร้าน/สาขา · นิติบุคคล (ชื่อ หรือเลขผู้เสียภาษีขึ้นต้น `0`) · ชื่อปลอม · เบอร์/อีเมลกลาง | **แยกออกจากชุดบุคคล** |
| Duplicate Matching | R0 CRMID เดียวกัน · R1 เลขบัตร · R2–R5 ชื่อ+เบอร์/อีเมล · R6 ข้ามภาษา · R7 ชื่อ+วันเกิด | **รวมแถวซ้ำ** |
| Review Queue | ชื่อ-นามสกุลเดียวกันแต่ไม่มีคีย์ยืนยัน → ไม่รวม ส่งตรวจ | – |

กฎทั้งหมดอยู่ใน [`scripts/ssot/cleanse.py`](../ssot/cleanse.py) — ใช้ร่วมกันทุก BU (CRM · ITEC · K2 · ITOS · D365)

**กติกาความปลอดภัย**
- ITEC-Rep read-only · credential จาก env `ITEC_USER` / `ITEC_PWD`
- ชื่อ เบอร์ อีเมล เลขบัตร ถูกแปลงเป็น hash ในหน่วยความจำ · ไม่เขียน PII ลงดิสก์ · ไม่เก็บ `CusCode` (มักเป็นเบอร์โทร)
- ตัวอย่างทุกตัว **mask** · ไฟล์ที่รันแล้วอยู่ใน `scripts/itec/_out/` (ไม่ขึ้น git)

In [ ]:
import sys, json, time, collections, gc, os, re, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
sys.path.insert(0, r'C:\Projects\my-first-project\scripts')
import db                              # credential อ่านจาก env ITEC_USER / ITEC_PWD
from ssot.cleanse import *             # กฎ cleansing กลาง
pd.set_option('future.infer_string', False)
pd.set_option('display.max_colwidth', 60)

CHUNK    = 100_000
MAX_ROWS = None        # ใส่ตัวเลข (เช่น 300_000) เพื่อทดสอบเร็ว · None = ทั้งตาราง
OUT      = r'C:\Projects\my-first-project\scripts\itec\_out'
cn = db.connect('itec', timeout=600)
print('connected')

## สิ่งที่ ITEC ต่างจาก CRM (จากการสำรวจ 2026-11-13)
| เรื่อง | ข้อเท็จจริง | ผลต่อการ clean |
|---|---|---|
| `CusCode` | เป็นเบอร์โทร 10 หลักราว ½ · ที่มี suffix (`.` `a` `_1` `-1`) เบอร์ตรงกับ `Tel` แค่ 9% | **ไม่ใช้ `CusCode` เป็นคีย์จับคู่** — suffix = เบอร์ถูกคนอื่นใช้ไปแล้ว |
| `CitizenNo` / `TaxNo` | `TaxNo` = `CitizenNo` 100% เมื่อมีทั้งคู่ · `TaxNo` มีเลขบัตรเพิ่มอีก ~12% | เลขบัตรบุคคล = `CitizenNo` ถ้าผ่าน checksum ไม่งั้น `TaxNo` |
| เลขขึ้นต้น `0` | เป็นเลขผู้เสียภาษีนิติบุคคล | ไม่ใช้เป็นเลขบัตรบุคคล · ใช้ช่วยจัดเป็นนิติบุคคล (ถ้า `NameTax` เป็นชื่อบริษัทและไม่มีเลขบัตรบุคคล) |
| `CRMID` | `M` + 12 หลัก = CRM `members.member_id` (เจอ 98.7%) | กฎ **R0**: หลาย `CusCode` ชี้ CRMID เดียว = คนเดียวกัน · และเป็นสะพานไป CRM |
| `Surname` | ไม่มีค่าว่าง แต่มี `.` / `-` แทน | ถือเป็นว่าง แล้วลองแยกจากช่องชื่อ |
| `Tel` | varchar(200) · บางแถวมีหลายเบอร์ | ใช้เบอร์แรก (นับไว้) |

## ทดลองกฎกับข้อมูลสมมติ (ไม่ใช่ข้อมูลลูกค้าจริง)

In [ ]:
def fake_id(base12):       # สร้างเลข 13 หลักที่ checksum ถูก สำหรับ demo
    s = sum(int(x) * (13 - i) for i, x in enumerate(base12)); return base12 + str((11 - s % 11) % 10)

demo = pd.DataFrame({'Name':    ['คุณสมชาย', 'น.ส.สุดา ใจงาม', 'Mr.Somchai', 'ลูกค้าทั่วไป', 'บริษัท ตัวอย่าง', 'สมหญิง'],
                     'Surname': ['ใจดี', '.', 'Jaidee', '-', 'จำกัด', 'ศรีสุข'],
                     'Tel':     ['081-234-5679', '081-234-5679, 02-123-4567', '+66 81 234 5679', '0999999999', '02 123 4567', '๐๘๑๒๓๔๕๖๗๙'],
                     'CitizenNo': [fake_id('110170000001'), '', fake_id('110170000001')[:12] + '0', '', '', ''],
                     'TaxNo':     ['', fake_id('310170000002'), '', '', fake_id('010550000003'), '1111111111111']})
nm = clean_names(demo.Name, demo.Surname); ph = clean_phone(demo.Tel)
cd, cv, ck = thai_id(demo.CitizenNo); td, tv, tk = thai_id(demo.TaxNo)
pd.DataFrame({'Name | Surname': demo.Name + ' | ' + demo.Surname, 'title': nm['title'], 'first': nm['first'], 'last': nm['last'],
              'ชื่อปลอม': nm['junk'], 'นิติ (ชื่อ)': nm['juristic'],
              'Tel': demo.Tel.map(lambda s: re.sub(r'(?<=\d{3})\d', 'x', s)), 'เบอร์หลัง clean': ph['norm'].map(mask_phone), 'สถานะเบอร์': ph['status'],
              'CitizenNo ใช้ได้': cv, 'TaxNo ชนิด': pd.Series(tk).map({0: '-', 1: 'บุคคล', 2: 'นิติบุคคล'})})

## Extract + Standardize + Validate + Classify — ทีละ 100,000 แถว
ในแต่ละ chunk: clean ทุกคอลัมน์ → เก็บเฉพาะ **hash + ธง** ลง array · นับสถิติแต่ละกฎ · เก็บตัวอย่างแบบ mask กฎละไม่กี่แถว

สถานะแถว: `0` บุคคล · `1` ไม่มีข้อมูลระบุตัว (ชื่อใช้ไม่ได้ + ไม่มีเบอร์/อีเมล/เลขบัตร) · `4` บัญชีร้าน/สาขา · `2` นิติบุคคล · `3` ชื่อปลอม (แต่มีช่องทางติดต่อ)

In [ ]:
SQL = ("SELECT TOP (?) CusCode, Name, Surname, Tel, Email, CitizenNo, TaxNo, NameTax, HBD, Gender, CusCompany, CRMID "
       "FROM dbo.Customer2 WITH (NOLOCK) WHERE CusCode > CAST(? AS varchar(20)) ORDER BY CusCode")
EX_N = 6
def add_ex(ex, k, rows):
    lst = ex.setdefault(k, [])
    if len(lst) < EX_N: lst.extend(rows[:EX_N - len(lst)])
def mask_raw(s):           # เก็บโครงสร้าง (ขีด/ช่องว่าง/จุลภาค) · ตัวเลขเหลือ 3 ตัวแรกของทั้งช่อง
    n = 0; out = []
    for ch in str(s):
        if ch.isdigit(): n += 1; out.append(ch if n <= 3 else 'x')
        else: out.append(ch)
    return ''.join(out)[:40]

def fetch(last):
    global cn
    for attempt in range(6):                       # VPN/เครือข่ายหลุด → ต่อใหม่แล้วดึง chunk เดิมซ้ำ
        try:
            cur = cn.cursor(); cur.execute(SQL, CHUNK, last)
            return cur.fetchall(), [c[0] for c in cur.description]
        except Exception as err:
            print(f'  retry {attempt + 1}: {str(err)[:80]}'); time.sleep(30 * (attempt + 1))
            try: cn.close()
            except Exception: pass
            cn = db.connect('itec', timeout=600)
    raise RuntimeError('ดึง chunk ไม่ได้หลังลอง 6 ครั้ง')

KEYS = dict(status='u1', lang='u1', name='u8', phone='i8', email='u8', birth='i4', init='u2', company='?', idno='u8', link='u8')

def run_itec():
    cnt = collections.Counter(); ex = {}; junk_vals = collections.Counter(); phone_bad = collections.Counter()
    parts = {k: [] for k in KEYS}; starts = []           # starts = (offset, CusCode แรกของ chunk) ใช้ดึงตัวอย่างกลับ · อยู่ในหน่วยความจำเท่านั้น
    last = ''; rows = 0; t0 = time.time()
    while True:
        recs, cols = fetch(last)
        if not recs: break
        df = pd.DataFrame.from_records(recs, columns=cols); del recs
        starts.append((rows, df.CusCode.iat[0])); last = df.CusCode.iat[-1]
        n = len(df); rows += n
        nm = clean_names(df.Name, df.Surname)
        ph = clean_phone(df.Tel); em = clean_email(df.Email); bd = clean_birth(df.HBD)
        cd, cv, ck = thai_id(df.CitizenNo); td, tv, tk = thai_id(df.TaxNo)
        pid_d = pd.Series(np.where(ck == 1, cd, np.where(tk == 1, td, '')), dtype=object)
        has_pid = (ck == 1) | (tk == 1)
        idno = id_hash(pid_d, has_pid)
        tax_jur = (tk == 2) | (ck == 2)
        ntx = df.NameTax.fillna('').astype(str)
        jur_name = nm['juristic'].to_numpy()
        jur_tax = tax_jur & ~has_pid & ntx.str.contains(JURISTIC, case=False).to_numpy()
        jur = jur_name | jur_tax
        junk = (nm['junk'] != '').to_numpy()
        noid = junk & (ph['status'] != 'ok').to_numpy() & (em['status'] != 'ok').to_numpy() & ~has_pid
        intl = nm['internal'].to_numpy()
        status = np.select([noid, intl, jur, junk], [1, 4, 2, 3], 0).astype('u1')
        crm = df.CRMID.fillna('').astype(str).str.strip(); crm_ok = crm.str.fullmatch(r'M\d{12}')
        link = np.where(crm_ok, pd.to_numeric(crm.str[1:].where(crm_ok, '0'), errors='coerce'), 0).astype('u8')
        V = dict(status=status, lang=nm['lang'], name=name_hash(nm['key'], junk), phone=ph['val'], email=em['hash'],
                 birth=bd['val'], init=init_class(nm['first'], nm['last'], nm['lang']), company=em['company'].to_numpy(),
                 idno=idno, link=link)
        for k, t in KEYS.items(): parts[k].append(np.asarray(V[k]).astype(t))
        # ── counters ──
        cnt['rows'] += n
        cnt['title'] += int(nm['title_in'].sum()); cnt['split'] += int(nm['split'].sum())
        cnt['dup_last'] += int(nm['dup_last'].sum()); cnt['middle'] += int(nm['middle'].sum())
        cnt['last_placeholder'] += int(nm['last_placeholder'].sum())
        for k, v in zip(*np.unique(nm['lang'], return_counts=True)): cnt[f'lang_{k}'] += int(v)
        cnt['th_first_en_last'] += int(((nm['lang'] == 1) & (nm['last_lang'] == 2)).sum())
        cnt['en_first_th_last'] += int(((nm['lang'] == 2) & (nm['last_lang'] == 1)).sum())
        cnt['same_first_last'] += int(nm['same_fl'].sum())
        for k, v in nm['junk'][nm['junk'] != ''].value_counts().items(): cnt[f'junk_{k}'] += int(v)
        for k, v in ph['status'].value_counts().items(): cnt[f'phone_{k}'] += int(v)
        cnt['phone_fixed'] += int(ph['fixed'].sum()); cnt['phone_multi'] += int(ph['multi'].sum()); cnt['phone_mobile'] += int(ph['mobile'].sum())
        for k, v in em['status'].value_counts().items(): cnt[f'email_{k}'] += int(v)
        cnt['email_domain_fixed'] += int(em['fixed_domain'].sum()); cnt['email_company'] += int(em['company'].sum())
        cnt['birth_present'] += int((bd['val'] > 0).sum()); cnt['birth_be_fixed'] += int(bd['be_fixed'].sum())
        cnt['birth_invalid'] += int(bd['invalid'].sum()); cnt['birth_jan1'] += int(bd['jan1'].sum())
        cnt['cit_present'] += int((cd != '').sum()); cnt['cit_valid'] += int(cv.sum()); cnt['cit_person'] += int((ck == 1).sum())
        cnt['tax_present'] += int((td != '').sum()); cnt['tax_valid'] += int(tv.sum())
        cnt['tax_person'] += int((tk == 1).sum()); cnt['tax_juristic'] += int((tk == 2).sum())
        both = (ck == 1) & (tk == 1)
        cnt['tax_eq_cit'] += int((both & (cd.to_numpy() == td.to_numpy())).sum()); cnt['tax_ne_cit'] += int((both & (cd.to_numpy() != td.to_numpy())).sum())
        cnt['id_from_tax_only'] += int(((ck != 1) & (tk == 1)).sum()); cnt['pid_any'] += int(has_pid.sum())
        cnt['juristic_by_name'] += int((jur_name & (status == 2)).sum()); cnt['juristic_by_taxid'] += int((jur_tax & ~jur_name & (status == 2)).sum())
        cnt['taxjur_person_kept'] += int((tax_jur & (status == 0)).sum())
        cnt['junk_with_pid'] += int((junk & has_pid).sum())
        cnt['crmid_present'] += int((crm != '').sum()); cnt['crmid_valid'] += int(crm_ok.sum())
        cnt['cuscompany_present'] += int((df.CusCompany.fillna('').astype(str).str.strip() != '').sum())
        for k, v in zip(*np.unique(status, return_counts=True)): cnt[f'status_{k}'] += int(v)
        for (g, s), v in pd.Series(1, index=pd.MultiIndex.from_arrays([df.Gender.to_numpy(), status])).groupby(level=[0, 1]).size().items():
            cnt[f'gender_{g}_status_{s}'] += int(v)
        fo, lo_ = df.Name.fillna('').astype(str), df.Surname.fillna('').astype(str)
        phone_bad.update(pattern(x) for x in df.Tel[(ph['status'] == 'invalid').to_numpy()])
        junk_vals.update(mask_digits(x)[:30] for x in (fo.str.strip() + ' ' + lo_.str.strip()).str.strip()[junk])
        # ── examples (masked) ──
        I = df.index
        m = nm['title_in'];             add_ex(ex, 'ตัดคำนำหน้า', [(nm['title'][i] + mask_tok(nm['first'][i]) + ' | ' + mask_tok(lo_[i]), mask_tok(nm['first'][i])) for i in I[m][:EX_N]])
        m = nm['split'];                add_ex(ex, 'แยกชื่อ-นามสกุลจากช่องเดียว', [(mask_tok(fo[i]) + ' | ' + lo_[i][:3], mask_tok(nm['first'][i]) + ' | ' + mask_tok(nm['last'][i])) for i in I[m][:EX_N]])
        m = nm['dup_last'];             add_ex(ex, 'นามสกุลซ้ำในช่องชื่อ', [(mask_tok(fo[i]) + ' | ' + mask_tok(lo_[i]), mask_tok(nm['first'][i]) + ' | ' + mask_tok(nm['last'][i])) for i in I[m][:EX_N]])
        m = ph['multi'];                add_ex(ex, 'หลายเบอร์ในช่อง Tel → ใช้เบอร์แรก', [(mask_raw(df.Tel[i]), mask_phone(ph['norm'][i])) for i in I[m][:EX_N]])
        m = ph['fixed'] & ~ph['multi']; add_ex(ex, 'แก้รูปแบบเบอร์', [(mask_raw(df.Tel[i]), mask_phone(ph['norm'][i])) for i in I[m][:EX_N]])
        m = em['fixed_domain'];         add_ex(ex, 'แก้โดเมนอีเมล', [(mask_email(str(df.Email[i]).strip().lower()), mask_email(em['norm'][i])) for i in I[m][:EX_N]])
        m = pd.Series(status == 4, index=I); add_ex(ex, 'บัญชีร้าน/สาขา', [(mask_org(fo[i] + ' ' + lo_[i]), '→ แยกออก (ไม่ใช่ลูกค้า)') for i in I[m][:EX_N]])
        m = pd.Series(jur_name & (status == 2), index=I); add_ex(ex, 'นิติบุคคล (ชื่อ)', [(mask_org(fo[i] + ' ' + lo_[i]), '→ ชุดนิติบุคคล') for i in I[m][:EX_N]])
        m = pd.Series(jur_tax & ~jur_name & (status == 2), index=I); add_ex(ex, 'นิติบุคคล (เลขผู้เสียภาษี 0… + NameTax)', [(mask_tok(fo[i] + ' ' + lo_[i]) + ' · ' + mask_org(ntx[i]), '→ ชุดนิติบุคคล') for i in I[m][:EX_N]])
        if len(starts) % 25 == 0: print(f'  {rows:,} rows · {time.time() - t0:,.0f}s')
        if MAX_ROWS and rows >= MAX_ROWS: break
        if n < CHUNK: break
    a = {}
    for k in KEYS:
        a[k] = np.concatenate(parts[k]); parts[k] = None
    gc.collect()
    print(f'ITEC-Rep Customer2: {rows:,} rows · {time.time() - t0:,.0f}s')
    return dict(cnt=cnt, ex=ex, junk_vals=junk_vals, phone_bad=phone_bad, arr=a, starts=starts)

In [ ]:
R = run_itec(); a = R['arr']; cnt = R['cnt']

## Shared contacts · Duplicate matching · Review queue
- เบอร์/อีเมลเดียวที่มีชื่อต่างกัน ≥ 5 ชื่อ = เบอร์ร้าน/พนักงานกรอกแทน → ไม่ใช้เป็นคีย์จับคู่
- อีเมล @comseven.com: 1 อีเมล 1 ชื่อ = พนักงานที่เป็นลูกค้า (เก็บ) · หลายชื่อ = กรอกแทน (ลบอีเมล)

| กฎ | คีย์ | ความมั่นใจ |
|---|---|---|
| R0 | `CRMID` เดียวกัน (หลาย `CusCode` ชี้สมาชิก CRM คนเดียว) | แน่นอน |
| R1 | เลขบัตรบุคคล (`CitizenNo` / `TaxNo` ผ่าน checksum · ไม่ขึ้นต้น 0) | สูงสุด |
| R2–R5 | ชื่อไทย/อังกฤษ + เบอร์/อีเมล | สูง |
| R6 | ข้ามภาษา TH↔EN (เบอร์/อีเมลเดียวกัน + อักษรต้นชื่อตรงกัน) | กลาง-สูง |
| R7 | ชื่อ + วันเกิด (ไม่นับ 1 ม.ค.) | กลาง-สูง |

In [ ]:
t0 = time.time()
shared_contacts(a, cnt)
R['steps'], R['n_person'], R['unique'], R['edges_r6'] = dedup(a, r0_label='R0 CRMID เดียวกัน')
R['review'] = review_same_name(a)
print(f'{time.time() - t0:,.0f}s')
pd.DataFrame(R['steps']).style.format({'edges': '{:,}', 'merged': '{:,}', 'unique_after': '{:,}'})

## ผล Standardize · Validate · Classify

In [ ]:
LBL = [('rows', 'แถวทั้งหมด'),
       ('title', 'Name · มีคำนำหน้าในช่องชื่อ → ตัดออก'), ('split', 'Name · ชื่อ+นามสกุลอยู่ช่องเดียว → แยก'),
       ('last_placeholder', 'Name · นามสกุลเป็นแค่ . / - (ถือว่าว่าง)'),
       ('dup_last', 'Name · นามสกุลซ้ำในช่องชื่อ → ตัด'), ('middle', 'Name · ยังมีช่องว่างในชื่อ (ชื่อกลาง/ชื่อยาว)'),
       ('lang_1', 'Name · ชื่อภาษาไทย'), ('lang_2', 'Name · ชื่อภาษาอังกฤษ'), ('lang_3', 'Name · ไทยปนอังกฤษในช่องเดียว'),
       ('th_first_en_last', 'Name · ชื่อไทย + นามสกุลอังกฤษ'), ('en_first_th_last', 'Name · ชื่ออังกฤษ + นามสกุลไทย'),
       ('phone_ok', 'Contact · เบอร์ใช้ได้'), ('phone_mobile', 'Contact   ↳ เป็นมือถือ'), ('phone_fixed', 'Contact   ↳ แก้รูปแบบแล้วใช้ได้'),
       ('phone_multi', 'Contact   ↳ หลายเบอร์ในช่องเดียว (ใช้เบอร์แรก)'), ('phone_empty', 'Contact · เบอร์ว่าง'),
       ('phone_invalid', 'Contact · เบอร์ผิดรูป'), ('phone_dummy', 'Contact · เบอร์ปลอม (เลขซ้ำ/เรียง)'),
       ('email_ok', 'Contact · อีเมลใช้ได้'), ('email_domain_fixed', 'Contact   ↳ แก้โดเมนสะกดผิด'), ('email_empty', 'Contact · อีเมลว่าง'),
       ('email_invalid', 'Contact · อีเมลผิดรูป'), ('email_dummy', 'Contact · อีเมลปลอม'),
       ('birth_present', 'Contact · วันเกิดใช้ได้'), ('birth_be_fixed', 'Contact   ↳ แก้ปี พ.ศ.'), ('birth_jan1', 'Contact · วันเกิด 1 ม.ค. (ไม่ใช้จับคู่)'),
       ('cit_present', 'ID · มี CitizenNo'), ('cit_valid', 'ID   ↳ ผ่าน checksum'), ('cit_person', 'ID   ↳ เป็นเลขบัตรบุคคล'),
       ('tax_present', 'ID · มี TaxNo'), ('tax_valid', 'ID   ↳ ผ่าน checksum'), ('tax_person', 'ID   ↳ เลขบัตรบุคคล'),
       ('tax_juristic', 'ID   ↳ เลขผู้เสียภาษีนิติบุคคล (ขึ้นต้น 0)'), ('tax_eq_cit', 'ID · TaxNo = CitizenNo (มีทั้งคู่)'),
       ('tax_ne_cit', 'ID · TaxNo ≠ CitizenNo (มีทั้งคู่)'), ('id_from_tax_only', 'ID · ได้เลขบัตรจาก TaxNo อย่างเดียว'),
       ('pid_any', 'ID · มีเลขบัตรบุคคลที่ใช้ได้ (รวม)'),
       ('crmid_present', 'CRM · มี CRMID'), ('crmid_valid', 'CRM   ↳ รูปแบบ M+12 หลัก'),
       ('status_1', 'Class ✂ ไม่มีข้อมูลระบุตัว'), ('status_4', 'Class ✂ บัญชีร้าน/สาขา'),
       ('status_2', 'Class ✂ นิติบุคคล → แยกชุด'), ('juristic_by_name', 'Class   ↳ จากชื่อ'), ('juristic_by_taxid', 'Class   ↳ จากเลขผู้เสียภาษี 0… + NameTax'),
       ('status_3', 'Class ✂ ชื่อปลอม/placeholder (มีช่องทางติดต่อ)'), ('junk_with_pid', 'Class   ↳ ชื่อปลอมแต่มีเลขบัตร (รวมทุกสถานะ)'),
       ('status_0', 'Class ✔ บุคคลที่เข้า dedup'), ('taxjur_person_kept', 'Class   ↳ TaxNo นิติฯ แต่ชื่อเป็นบุคคล (เก็บเป็นบุคคล)'),
       ('same_first_last', 'Flag · ชื่อ = นามสกุล'), ('email_company', 'Flag · อีเมล @comseven.com'),
       ('company_1name', 'Flag   ↳ 1 อีเมล 1 ชื่อ = พนักงาน'), ('company_multi', 'Flag   ↳ 1 อีเมลหลายชื่อ = กรอกแทน (ลบอีเมล)'),
       ('shared_phone_rows', 'Flag · แถวที่ใช้เบอร์กลาง (≥ 5 ชื่อ)'), ('shared_phone_keys', 'Flag   ↳ จำนวนเบอร์กลาง'),
       ('shared_email_rows', 'Flag · แถวที่ใช้อีเมลกลาง (≥ 5 ชื่อ)'), ('shared_email_keys', 'Flag   ↳ จำนวนอีเมลกลาง'),
       ('cuscompany_present', 'Info · มีค่า CusCompany')]
T = pd.DataFrame({'ITEC-Rep': [cnt.get(c, 0) for c, _ in LBL]}, index=[l for _, l in LBL])
T['%'] = T['ITEC-Rep'] / cnt['rows'] * 100
T.style.format({'ITEC-Rep': '{:,.0f}', '%': '{:.1f}'})

### Gender × สถานะ — ดูว่ารหัส Gender 3/4 คืออะไร (ทดสอบ 300k แถวแรก: Gender 3/4 ส่วนใหญ่เป็นนิติบุคคล)
`status`: 0 บุคคล · 1 ไม่มีข้อมูลระบุตัว · 2 นิติบุคคล · 3 ชื่อปลอม · 4 บัญชีร้าน

In [ ]:
G = pd.Series({tuple(map(int, k.split('_')[1::2])): v for k, v in cnt.items() if k.startswith('gender_')})
G.unstack(fill_value=0).rename_axis(index='Gender', columns='status').style.format('{:,.0f}')

### เหตุผลที่ชื่อถูกมองว่าไม่ใช่ชื่อจริง

In [ ]:
display(pd.Series({c[5:]: v for c, v in cnt.items() if c.startswith('junk_') and c != 'junk_with_pid'}).sort_values(ascending=False).to_frame('แถว').style.format('{:,.0f}'))
print('ค่าชื่อปลอมที่เจอบ่อยสุด (ตัวเลขถูก mask · แสดงเฉพาะค่าที่ซ้ำ ≥ 20 ครั้ง = ไม่ระบุตัวคน):')
pd.DataFrame([(v, c) for v, c in R['junk_vals'].most_common(40) if c >= 20], columns=['ค่า', 'แถว'])

### ตัวอย่างแต่ละกฎ (mask แล้ว)

In [ ]:
for rule, rows in R['ex'].items():
    if rows: display(pd.DataFrame(rows, columns=['ก่อน', 'หลัง']).assign(กฎ=rule)[['กฎ', 'ก่อน', 'หลัง']])

### เบอร์ผิดรูปเป็นแบบไหน (รูปแบบ · 9 = ตัวเลข · ไม่แสดงค่าจริง)

In [ ]:
pd.DataFrame(R['phone_bad'].most_common(15), columns=['รูปแบบ Tel', 'แถว']).style.format({'แถว': '{:,.0f}'})

### อีเมล @comseven.com — จำนวนชื่อไม่ซ้ำต่ออีเมล (5 = 5 ชื่อขึ้นไป)

In [ ]:
pd.Series(cnt.get('company_dist', {}), name='อีเมล').rename_axis('ชื่อต่ออีเมล')

### ตัวอย่างกฎข้ามภาษา R6 (ดึง 6 กลุ่มกลับมาดู · mask แล้ว)
ดึงแถวกลับด้วยตำแหน่งใน chunk · ถ้าข้อมูลเปลี่ยนระหว่างรัน (ITEC-Rep อัปเดตตลอด) ชื่อจะไม่ตรง hash → ข้ามแถวนั้น

In [ ]:
OFFS = np.array([s[0] for s in R['starts']]); STARTS = [s[1] for s in R['starts']]
def fetch_row(i):
    k = int(np.searchsorted(OFFS, i, side='right') - 1)
    cur = cn.cursor()
    cur.execute("SELECT Name, Surname, Tel, Email FROM dbo.Customer2 WITH (NOLOCK) WHERE CusCode >= CAST(? AS varchar(20)) "
                "ORDER BY CusCode OFFSET ? ROWS FETCH NEXT 1 ROWS ONLY", STARTS[k], int(i - OFFS[k]))
    return cur.fetchone()
e = R['edges_r6']; rows = []
if e is not None and len(e):
    for g, p in enumerate(e[np.random.default_rng(7).choice(len(e), size=min(6, len(e)), replace=False)]):
        for i in p:
            r = fetch_row(int(i))
            if r is None: continue
            nm = clean_names(pd.Series([r[0]]), pd.Series([r[1]]))
            if name_hash(nm['key'], np.array([False]))[0] != a['name'][i]: continue
            rows.append((g + 1, mask_tok(nm['first'][0]), mask_tok(nm['last'][0]), mask_phone(clean_phone(pd.Series([r[2]]))['norm'][0]), mask_email((r[3] or '').strip().lower())))
pd.DataFrame(rows, columns=['กลุ่ม', 'ชื่อ', 'นามสกุล', 'เบอร์', 'อีเมล'])

## Review queue — ชื่อ-นามสกุลเดียวกันแต่ไม่มีคีย์ยืนยัน
- 2–3 กลุ่มต่อชื่อ → อาจเป็นคนเดียวที่เปลี่ยนเบอร์ → รอยืนยันด้วยเลขบัตรจากฐานอื่น (K2/ITOS) ตอนรวมข้าม BU
- ≥ 4 กลุ่มต่อชื่อ → ชื่อโหล → ไม่รวม

In [ ]:
pd.Series(R['review']).to_frame('ITEC-Rep').style.format('{:,.0f}')

## Match readiness — ลูกค้าไม่ซ้ำแต่ละคนมีคีย์อะไรไว้ map กับฐานอื่น
นับต่อ **ลูกค้า 1 คน** (กลุ่มหลัง dedup) — มีคีย์นั้นอย่างน้อย 1 แถวในกลุ่ม

In [ ]:
person = a['status'] == 0
mobile = a['phone_m'] >= 10**8                     # เบอร์ 0[689]xxxxxxxx → ตัวเลข 9 หลักขึ้นไป
U = R['unique']
RD = {'เลขบัตรบุคคล (R1 → K2 · ITOS · D365)': cluster_any(a, a['idno'] != 0),
      'มือถือ': cluster_any(a, mobile),
      'อีเมล': cluster_any(a, a['email_m'] != 0),
      'CRMID (สะพานไป CRM)': cluster_any(a, a['link'] != 0),
      'วันเกิด': cluster_any(a, a['birth'] != 0),
      'เลขบัตร หรือ มือถือ': cluster_any(a, (a['idno'] != 0) | mobile),
      'มีอย่างน้อย 1 คีย์ (บัตร/มือถือ/อีเมล)': cluster_any(a, (a['idno'] != 0) | mobile | (a['email_m'] != 0))}
RD = pd.DataFrame({'ลูกค้า': RD}); RD['%'] = RD['ลูกค้า'] / U * 100
# ลูกค้า ITEC 1 คนที่ผูกกับสมาชิก CRM หลาย member_id = สัญญาณว่า CRM มีสมาชิกซ้ำ
lab1 = (a['cluster'].astype('u8') + 1)
dc = distinct_per_key(lab1, a['link'], person & (a['link'] != 0))
R['crm_multi'] = dict(crmid_unique=int(len(np.unique(a['link'][person & (a['link'] != 0)]))),
                      itec_person_multi_crmid=int(len(np.unique(lab1[dc >= 2]))))
display(RD.style.format({'ลูกค้า': '{:,.0f}', '%': '{:.1f}'}))
print(f"ลูกค้าไม่ซ้ำ {U:,} คน · CRMID ไม่ซ้ำที่อ้างถึง {R['crm_multi']['crmid_unique']:,} · ลูกค้า ITEC ที่ผูกกับ CRMID ≥ 2 ตัว {R['crm_multi']['itec_person_multi_crmid']:,} คน")
R['readiness'] = {k: int(v) for k, v in RD['ลูกค้า'].items()}

## Funnel สรุป

In [ ]:
f = [('แถวดิบ', cnt['rows']), ('✂ ไม่มีข้อมูลระบุตัว', -cnt.get('status_1', 0)), ('✂ บัญชีร้าน/สาขา', -cnt.get('status_4', 0)),
     ('✂ นิติบุคคล (แยกชุด)', -cnt.get('status_2', 0)), ('✂ ชื่อปลอม/placeholder', -cnt.get('status_3', 0))]
f += [('✂ ' + s['rule'], -s['merged']) for s in R['steps']]
f += [('= ลูกค้าบุคคลไม่ซ้ำ', R['unique'])]
for lab, v in f: print(f'   {lab:<34}{v:>14,}')
SUMMARY = {'IT': dict(name='ITEC-Rep Customer2 · ลูกค้า BNN/Studio7', funnel=f,
                      cnt={k: v for k, v in cnt.items() if not isinstance(v, dict)}, company_dist=cnt.get('company_dist', {}),
                      steps=R['steps'], review=R['review'], n_person=R['n_person'], unique=R['unique'],
                      readiness=R['readiness'], crm_multi=R['crm_multi'],
                      junk_top=[(v, c) for v, c in R['junk_vals'].most_common(25) if c >= 50],
                      phone_bad_top=R['phone_bad'].most_common(15)),
           '_run': time.strftime('%Y-%m-%d %H:%M'), '_max_rows': MAX_ROWS}
os.makedirs(OUT, exist_ok=True)
with open(os.path.join(OUT, 'itec_cleansing_summary.json'), 'w', encoding='utf-8') as fp:
    json.dump(SUMMARY, fp, ensure_ascii=False, indent=1, default=int)
print('saved summary (ตัวเลขรวมเท่านั้น ไม่มี PII)')

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Tahoma'
fig, ax = plt.subplots(figsize=(10, 6))
labs = [x[0] for x in f]; vals = [x[1] for x in f]; rem = np.cumsum(vals[:-1])
for i, v in enumerate(vals):
    if i in (0, len(vals) - 1): ax.barh(i, v, color='#1f4e79'); x = v
    else: ax.barh(i, -v, left=rem[i], color='#c0504d'); x = rem[i - 1]
    ax.text(x, i, f'  {abs(v):,}', va='center', fontsize=8)
ax.set_yticks(range(len(f))); ax.set_yticklabels(labs, fontsize=8); ax.invert_yaxis()
ax.set_title('ITEC-Rep Customer2'); ax.xaxis.set_visible(False)
for s in ['top', 'right', 'bottom']: ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

## หมายเหตุ / ข้อจำกัด
- `CusCode` ไม่ใช้จับคู่ (suffix = เบอร์ถูกคนอื่นใช้แล้ว) · ใช้เป็น `src_id` ของ ITEC อย่างเดียว
- เลขบัตรต้อง **HMAC** ก่อนเก็บลง SSOT (ในโน้ตบุ๊กนี้เป็น hash ในหน่วยความจำ ไม่ได้บันทึก)
- นิติบุคคลจากเลขผู้เสียภาษีจัดเฉพาะเมื่อ `NameTax` เป็นชื่อบริษัทและไม่มีเลขบัตรบุคคล — คนที่ขอใบกำกับในนามบริษัทแต่ชื่อลูกค้าเป็นคนยังเป็นบุคคล
- ตัวเลขเป็นของวันที่รัน · ITEC-Rep อัปเดต near real-time
- ขั้นถัดไป: map ITEC ↔ CRM ด้วย `CRMID` แล้วด้วยเลขบัตร/มือถือ+ชื่อ (ลำดับคีย์เดียวกับ IDENTITY layer)